# Right-half background subtraction and localisation

Compare the original image with mean subtraction and trailing sliding-window quantiles (`Q25`, `Q33`, and `Q50` median), using the same frame and Picasso settings for all five. Run the cells in order.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from picasso.localize import localize_frames
from numba import njit

ROOT = Path.cwd().parent
path = ROOT / "data" / "preprocessed_data" / "corrected.tif"
comparison_frame = 100  # zero-based; frame 49 is the first with a full window
window_size = 50
right_start = 428


## Sliding-window quantile (Q25, Q33, Q50)

For each pixel, the background is the specified quantile (`q` in `[0, 1]`, e.g. `0.25`, `1/3`, or `0.50` for the median) of the current frame and up to 49 preceding frames. The two-heap structure maintains the bottom $\lfloor q(n-1) \rfloor + 1$ values in a max-heap (`lower`) and the remaining values in a min-heap (`upper`), matching `np.quantile` with linear interpolation. The first run includes Numba compilation.


In [ ]:
from numba import njit


@njit
def _sift_down(heap, pixel, pos, size, is_max):
    while 2 * pos + 1 < size:
        child = 2 * pos + 1
        if child + 1 < size:
            if (heap[pixel, child + 1] > heap[pixel, child]) == is_max:
                child += 1
        if (heap[pixel, child] > heap[pixel, pos]) != is_max:
            break
        heap[pixel, pos], heap[pixel, child] = heap[pixel, child], heap[pixel, pos]
        pos = child


@njit
def _sift_up(heap, pixel, pos, is_max):
    while pos > 0:
        parent = (pos - 1) // 2
        if (heap[pixel, pos] > heap[pixel, parent]) != is_max:
            break
        heap[pixel, pos], heap[pixel, parent] = heap[pixel, parent], heap[pixel, pos]
        pos = parent


@njit
def _push(heap, pixel, size, value, is_max):
    heap[pixel, size] = value
    _sift_up(heap, pixel, size, is_max)


@njit
def _remove(heap, pixel, size, value, is_max):
    # Equal values are interchangeable, so removing any matching copy is sufficient.
    pos = 0
    while pos < size and heap[pixel, pos] != value:
        pos += 1
    if pos == size:
        raise ValueError("Expired value missing from heap")
    size -= 1
    if pos < size:
        heap[pixel, pos] = heap[pixel, size]
        if pos > 0 and (heap[pixel, pos] > heap[pixel, (pos - 1) // 2]) == is_max:
            _sift_up(heap, pixel, pos, is_max)
        else:
            _sift_down(heap, pixel, pos, size, is_max)


@njit
def _update_quantiles(lower, upper, history, values, slot, count, window_size, q):
    result = np.empty(values.size, dtype=np.float32)
    prev_n_lower = 0 if count == 0 else int(q * (count - 1)) + 1
    prev_n_upper = count - prev_n_lower
    new_count = count if count == window_size else count + 1
    idx = q * (new_count - 1)
    target_lower = int(idx) + 1
    frac = np.float32(idx - int(idx))

    for pixel in range(values.size):
        n_lower = prev_n_lower
        n_upper = prev_n_upper

        if count == window_size:
            old = history[slot, pixel]
            if old <= lower[pixel, 0]:
                _remove(lower, pixel, n_lower, old, True)
                n_lower -= 1
            else:
                _remove(upper, pixel, n_upper, old, False)
                n_upper -= 1

        value = values[pixel]
        if (n_lower > 0 and value <= lower[pixel, 0]) or (
            n_lower == 0 and (n_upper == 0 or value <= upper[pixel, 0])
        ):
            _push(lower, pixel, n_lower, value, True)
            n_lower += 1
        else:
            _push(upper, pixel, n_upper, value, False)
            n_upper += 1

        while n_lower > target_lower:
            middle = lower[pixel, 0]
            _remove(lower, pixel, n_lower, middle, True)
            n_lower -= 1
            _push(upper, pixel, n_upper, middle, False)
            n_upper += 1
        while n_lower < target_lower:
            middle = upper[pixel, 0]
            _remove(upper, pixel, n_upper, middle, False)
            n_upper -= 1
            _push(lower, pixel, n_lower, middle, True)
            n_lower += 1

        history[slot, pixel] = value
        if frac > 0.0 and n_upper > 0:
            result[pixel] = lower[pixel, 0] + frac * (upper[pixel, 0] - lower[pixel, 0])
        else:
            result[pixel] = lower[pixel, 0]
    return result


class SlidingWindowQuantile:
    """Streaming per-pixel trailing quantile, including the newest frame."""

    def __init__(self, window_size=50, quantile=0.5):
        if not isinstance(window_size, int) or window_size < 1:
            raise ValueError("window_size must be a positive integer")
        if not (0.0 <= quantile <= 1.0):
            raise ValueError("quantile must be between 0.0 and 1.0")
        self.window_size = window_size
        self.quantile = float(quantile)
        self.count = 0
        self.next_slot = 0
        self.shape = None

    def update(self, frame):
        frame = np.asarray(frame, dtype=np.float32)
        if frame.ndim != 2 or frame.size == 0 or not np.isfinite(frame).all():
            raise ValueError("frame must be a non-empty, finite 2D array")
        if self.shape is None:
            self.shape = frame.shape
            pixels = frame.size
            # One spare heap entry accommodates insertion before rebalancing.
            cap_lower = int(self.quantile * (self.window_size - 1)) + 2
            cap_upper = self.window_size - int(self.quantile * (self.window_size - 1)) + 1
            self.lower = np.empty((pixels, cap_lower), dtype=np.float32)
            self.upper = np.empty((pixels, cap_upper), dtype=np.float32)
            self.history = np.empty((self.window_size, pixels), dtype=np.float32)
        elif frame.shape != self.shape:
            raise ValueError(f"Expected frame shape {self.shape}, got {frame.shape}")

        q_frame = _update_quantiles(
            self.lower, self.upper, self.history, frame.ravel(),
            self.next_slot, self.count, self.window_size, self.quantile,
        )
        self.next_slot = (self.next_slot + 1) % self.window_size
        self.count = min(self.count + 1, self.window_size)
        return q_frame.reshape(self.shape)


class SlidingWindowMedian(SlidingWindowQuantile):
    """Streaming per-pixel trailing median (quantile=0.5)."""

    def __init__(self, window_size=50):
        super().__init__(window_size=window_size, quantile=0.5)

## Prepare the images

The mean is calculated over the selected right half. Negative residuals are clipped to zero for all subtraction methods (`Mean`, `Sliding Q25`, `Sliding Q33`, and `Sliding median (Q50)`). The original retains its unmodified pixel values.


In [ ]:
quantile_streams = {
    "Sliding Q25 subtracted": SlidingWindowQuantile(window_size=window_size, quantile=0.25),
    "Sliding Q33 subtracted": SlidingWindowQuantile(window_size=window_size, quantile=1 / 3),
    "Sliding median (Q50)": SlidingWindowMedian(window_size=window_size),
}
backgrounds = {}

with Image.open(path) as image:
    if not 0 <= comparison_frame < image.n_frames:
        raise ValueError(f"comparison_frame must be between 0 and {image.n_frames - 1}")
    for frame_index in range(comparison_frame + 1):
        image.seek(frame_index)
        original = np.array(image)[:, right_start:]
        for name, stream in quantile_streams.items():
            backgrounds[name] = stream.update(original)

original_float = original.astype(np.float32)
images = {
    "Original": original,
    "Mean subtracted": np.clip(original_float - original_float.mean(), 0, None),
    **{
        name: np.clip(original_float - bg, 0, None)
        for name, bg in backgrounds.items()
    },
}
sample_stream = next(iter(quantile_streams.values()))
print(f"Frame {comparison_frame}, right half: {original.shape}; "
      f"sliding window uses {sample_stream.count} frame(s)")


For visibility, the original has its own percentile-based contrast computed over non-zero pixels (since drift-correction border padding introduces ~1.1% zero pixels, which otherwise pulls the 1st percentile down to 0 and washes out the image); all residual images share a common scale. Adjust `original_percentiles` to change the original display. These limits affect the plots only, never the data sent to Picasso.


In [ ]:
original_percentiles = (5, 99.8)
nonzero_original = images["Original"][images["Original"] > 0]
original_limits = np.percentile(nonzero_original, original_percentiles)
residual_names = [name for name in images if name != "Original"]
residual_max = max(
    *(np.percentile(images[name], 99.8) for name in residual_names),
    1,
)
display_limits = {
    "Original": original_limits,
    **{name: (0, residual_max) for name in residual_names},
}

fig, axes = plt.subplots(1, len(images), figsize=(4.5 * len(images), 5.5), constrained_layout=True)
for ax, (name, frame) in zip(axes, images.items()):
    vmin, vmax = display_limits[name]
    ax.imshow(frame, cmap="gray", vmin=vmin, vmax=vmax)
    ax.set_title(name)
    ax.set_axis_off()
plt.show()


## Picasso localisation

Apply the same exploratory detection and fitting settings to all five images. Camera values are placeholders: use your camera calibration before interpreting photon counts.


In [ ]:
params = {
    "Box Size": 7,
    "Min. Net Gradient": 1000,
}

camera_info = {
    "Baseline": 0,
    "Sensitivity": 1.0,
    "Gain": 1,
    "Pixelsize": 117,  # nm; TIFF metadata says approximately 0.117 ?m
}

localisations = {}
for name, frame in images.items():
    localisations[name] = localize_frames(
        frame[np.newaxis, :, :],
        info=None,
        params=params,
        camera_info=camera_info,
        fitting_method="gausslq",
        threaded=False,
    )
    print(f"{name}: {len(localisations[name])} localisations")


In [ ]:
fig, axes = plt.subplots(1, len(images), figsize=(4.5 * len(images), 5.5), constrained_layout=True)
for ax, (name, frame) in zip(axes, images.items()):
    vmin, vmax = display_limits[name]
    ax.imshow(frame, cmap="gray", vmin=vmin, vmax=vmax)
    locs = localisations[name]
    ax.scatter(locs["x"], locs["y"], s=45, facecolors="none", edgecolors="red")
    ax.set_title(f"{name}: {len(locs)} localisations")
    ax.set_axis_off()
plt.show()


Localisation counts are an exploratory comparison, not a measure of accuracy. Background subtraction can change the net gradient, and the temporal median can suppress features that remain at the same pixel for much of the window. The placeholder camera baseline also affects fitted photon values, especially for the original image.
